In [170]:
#%pip install rapidfuzz

In [171]:
import pandas as pd
from rapidfuzz import process, fuzz
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [172]:
# Sample df1 with words to match
df_ACFull = pd.read_csv(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Autocare\20250731_Autocare_Attributes.csv")
df_ACFull

,PartTerminologyName,PAName
0,Car Cover,Length
1,Car Cover,Width
2,Car Cover,Material
3,Car Cover,Color
4,Car Cover,Vented
...,...,...
140523,Intercooler Fluid,Universal Or Specific Fit
140524,Intercooler Fluid,Grade Type
140525,Wheel Tape,Length
140526,Wheel Tape,Width


In [173]:
# Sample df1 with words to match
df_FBGFull = pd.read_csv(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\FBG_Data\FBG_SKU_Attributes.csv")
df_FBGFull

,Product Group,Source,BrandName,Brand,PartNumber,PartTerminologyName,Status,PAName,Attribute_Type,Value,AC_AttributeCount,Attribute_Value,Key,Parent-Child- Customer Brand
0,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Casting Number,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpCasting Number,Parent
1,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Fan Clutch Included,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpFan Clutch Included,Parent
2,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Grade Type,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpGrade Type,Parent
3,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Housing Material,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpHousing Material,Parent
4,Repair,PDM,BBDW_ASC,ASC,EWP1220,Engine Water Pump,Active,Hub Height,Autocare_Attribute,NaN,1.0,0.0,EWP1220Engine Water PumpHub Height,Parent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11428072,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Lens Width,FBG_Attribute,5.5,0.0,0.0,8261605Agriculture LightLens Width,Child
11428073,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Prop 65,FBG_Attribute,Yes,0.0,0.0,8261605Agriculture LightProp 65,Child
11428074,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Prop 65 Chemical,FBG_Attribute,DEHP,0.0,0.0,8261605Agriculture LightProp 65 Chemical,Child
11428075,Towing,Hitch Pro,Wesbar,Wesbar,8261605,Agriculture Light,Active,Prop 65 Warning,FBG_Attribute,WARNING: Cancer and Reproductive Harm – www.p6...,0.0,0.0,8261605Agriculture LightProp 65 Warning,Child


In [174]:
df_FBGFull=df_FBGFull[
    (df_FBGFull['Parent-Child- Customer Brand']=="Parent") & 
    (df_FBGFull['Attribute_Type']!="Autocare_Attribute")
    ].reset_index(drop=True)

In [175]:
df_FBGFull=df_FBGFull[['Product Group', 'Source', 'BrandName', 'Brand','PartTerminologyName', 'PAName','Parent-Child- Customer Brand']].drop_duplicates().reset_index(drop=True)

In [176]:
df_FBGFull.size

29722

In [177]:
PartNames=list(set(df_FBGFull['PartTerminologyName'].tolist()))

In [178]:
cols =['PartTerminologyName','Matching Autocare Attribute'] #,'Review_Mentions','Standards'
df_New = pd.DataFrame(columns=cols)
count=0

In [179]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 45

In [180]:
for i in range (len(PartNames)):
    df_FBGF=df_FBGFull[df_FBGFull['PartTerminologyName']==PartNames[i]].reset_index(drop=True)
    df_ACF=df_ACFull[df_ACFull['PartTerminologyName']==PartNames[i]]

    for j in range(len(df_FBGF)):
        #print(PartNames[i],df_FBGF['Cleaned Attributes'][j])
        df_New.at[count,'PartTerminologyName']=PartNames[i]
        df_New.at[count,'FBG Attributes']=df_FBGF['PAName'][j]
        df_New.at[count,'Product Group']= df_FBGF['Product Group'][j]
        df_New.at[count,'Source']= df_FBGF['Source'][j]
        df_New.at[count,'BrandName']= df_FBGF['BrandName'][j]
        df_New.at[count,'Brand']= df_FBGF['Brand'][j]
        df_New.at[count,'Matching Autocare Attribute']=get_matches(df_FBGF['PAName'][j],df_ACF['PAName'].tolist(), threshold=MATCH_THRESHOLD)
        count=count+1

In [181]:
df_New

,PartTerminologyName,Matching Autocare Attribute,FBG Attributes,Product Group,Source,BrandName,Brand
0,Engine Water Pump,"[Hub Hole Thread Size, Hub Hole Quantity, Outs...",Hub Hole Thread Diameter,Repair,PDM,BBDW_ASC,ASC
1,Engine Water Pump,"[Fan Clutch Included, Mounting Hardware Includ...",Gasket Or Seal Included,Repair,PDM,BBDW_ASC,ASC
2,Wheel Bearing and Hub Assembly,"[Bearing Type, Stud Thread Type, Drive Type, G...",ABS Sensor Type,Brakes,JNP,BPI,BPI
3,Wheel Bearing and Hub Assembly,"[Knuckle Bolt Circle Diameter, Flange Bolt Hol...",Steering Knuckle Flange Bolt Circle Diameter,Brakes,JNP,BPI,BPI
4,Wheel Bearing and Hub Assembly,"[Knuckle Pilot Hole Diameter, Knuckle Bolt Cir...",Steering Knuckle Flange Bolt Diameter,Brakes,JNP,BPI,BPI
...,...,...,...,...,...,...,...
4182,Turbocharger Vane Position Solenoid,[],FAQ_Q1,Steering/ Electronics,PDM,CNRD_CARDONE New,Cardone New
4183,Turbocharger Vane Position Solenoid,[],FAQ_Q2,Steering/ Electronics,PDM,CNRD_CARDONE New,Cardone New
4184,Turbocharger Vane Position Solenoid,[],FAQ_Q3,Steering/ Electronics,PDM,CNRD_CARDONE New,Cardone New
4185,Turbocharger Vane Position Solenoid,[],FAQ_Q4,Steering/ Electronics,PDM,CNRD_CARDONE New,Cardone New


In [169]:
df_New = df_New[df_New['Matching Autocare Attribute'].apply(str) != "[]"].reset_index(drop=True)
df_New = df_New.drop_duplicates(subset=['PartTerminologyName', 'FBG Attributes']).reset_index(drop=True)
df_New

,PartTerminologyName,Matching Autocare Attribute,FBG Attributes,Product Group,Source,BrandName,Brand
0,Engine Water Pump,"[Hub Hole Thread Size, Hub Hole Quantity, Outs...",Hub Hole Thread Diameter,Repair,PDM,BBDW_ASC,ASC
1,Engine Water Pump,"[Fan Clutch Included, Mounting Hardware Includ...",Gasket Or Seal Included,Repair,PDM,BBDW_ASC,ASC
2,Wheel Bearing and Hub Assembly,"[Bearing Type, Stud Thread Type, Drive Type, G...",ABS Sensor Type,Brakes,JNP,BPI,BPI
3,Wheel Bearing and Hub Assembly,"[Knuckle Bolt Circle Diameter, Flange Bolt Hol...",Steering Knuckle Flange Bolt Circle Diameter,Brakes,JNP,BPI,BPI
4,Wheel Bearing and Hub Assembly,"[Knuckle Pilot Hole Diameter, Knuckle Bolt Cir...",Steering Knuckle Flange Bolt Diameter,Brakes,JNP,BPI,BPI
...,...,...,...,...,...,...,...
988,Electric Fuel Pump Repair Kit,"[Fuel Type, Pump Pressure, Engine Fuel Type]",Pump Type,Repair,PDM,BCQS_Carter,Carter
989,Electric Fuel Pump Repair Kit,[Connector Color],Color,Repair,PDM,BCQS_Carter,Carter
990,Diesel Particulate Filter (DPF),"[Inlet Mounting Hole Quantity, Outlet Mounting...",Is Or Contains A Battery,Steering/ Electronics,PDM,CPCX_CARDONE Reman,Cardone Reman
991,Diesel Particulate Filter (DPF),[Sensor Port Count],Product Condition,Steering/ Electronics,PDM,CPCX_CARDONE Reman,Cardone Reman


In [183]:
df_New.to_excel(r'C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\FBG_Data\Phase-1\20250805_approximate_matches.xlsx', index=False)

## Misc Code


In [ ]:

# Sample df1 with words to match
df_FB = pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Potential Match.xlsx",sheet_name="FB")
df_FB

In [ ]:
df_AC = pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Potential Match.xlsx",sheet_name="AC")
df_AC

In [ ]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 80


In [ ]:

# Apply the function to df1
df_FB['approx_matches'] = df_FB['Cleaned FBG Attribute'].apply(
    lambda w: get_matches(w, df_AC['AC Attribute'].tolist(), threshold=MATCH_THRESHOLD)
)

df_FB=df_FB.drop_duplicates(inplace=True).rest_index(drop=True)


In [ ]:
df_FB.to_excel(r'C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\202approximate_matches.xlsx', index=False)